In [4]:
import pandas as pd

df = pd.read_csv("../results/all_genes_results.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(12548, 9)
['probe', 'log2FC', 'p_value', 'p_adj', 'neg_log_padj', 'status', 'gene', 'mean_expr', 'nef_log_padj']


,probe,log2FC,p_value,p_adj,neg_log_padj,status,gene,mean_expr,nef_log_padj
0,220951_s_at,-0.005406,9.343841e-01,9.569482e-01,0.019112,NS,A1CF,7.702939,0.019112
1,217757_at,-1.358073,1.286247e-22,7.961513e-21,20.099004,Down,A2M,12.666776,20.099004
2,219488_at,0.002289,9.659079e-01,9.780943e-01,0.009619,NS,A4GALT,5.709294,0.009619
3,221131_at,-0.057387,2.317204e-02,4.776968e-02,1.320848,NS,A4GNT,6.290016,1.320848
4,218075_at,0.109924,1.768453e-02,3.765905e-02,1.424131,NS,AAAS,9.082229,1.424131


In [5]:
print(len(df))
print(df['gene'].nunique())

12548
12548


In [6]:
for g in ["NEK2", "AGER"]:
    row = df[df["gene"] == g]
    print(g, "log2FC =", row["log2FC"].values)

NEK2 log2FC = [1.5153654]
AGER log2FC = [-4.41746953]


In [9]:
panel = ["HLA-A","HLA-B","HLA-C","B2M","TAP1","TAP2",
         "TAPBP","PSMB8","PSMB9","NLRC5","IRF1","STAT1"]
sub = df[df["gene"].isin(panel)]
print(sub.shape)
print(sub[["gene", "log2FC", "p_adj"]])

(11, 9)
        gene    log2FC     p_adj
945      B2M -0.200713  0.000364
4768   HLA-A -0.068573  0.380888
4769   HLA-B -0.156182  0.052910
4770   HLA-C -0.169329  0.002009
5290    IRF1 -0.536591  0.000015
8662   PSMB8  0.250677  0.028050
8663   PSMB9  0.040530  0.827216
10619  STAT1  0.507444  0.000041
10822   TAP1  0.176616  0.216134
10823   TAP2  0.170167  0.036731
10824  TAPBP  0.096526  0.313129


In [11]:
found = set(sub["gene"])
missing = set(panel) - found
print("The number genes we found:", len(found))
print("The number genes we missed:", missing)

The number genes we found: 11
The number genes we missed: {'NLRC5'}


In [12]:
cond_fdr = sub["p_adj"] < 0.001            
cond_fc  = sub["log2FC"].abs() > 0.585
passes = (sub["p_adj"] < 0.001) & (sub["log2FC"].abs() > 0.585)
print("The number genes passing 2 conditions:", passes.sum())

The number genes passing 2 conditions: 0


In [13]:

refs = ["CD8A", "PTPRC", "CD68", "SPP1", "MMP12", "MMP1"]
ref_sub = df[df["gene"].isin(refs)]
print(ref_sub[["gene", "log2FC", "p_adj"]])
print("Miss", set(refs) - set(ref_sub["gene"]))  

        gene    log2FC         p_adj
1753    CD8A -0.042289  7.006340e-01
6621    MMP1  2.862036  1.649280e-14
6624   MMP12  2.377325  2.587063e-18
8763   PTPRC -0.655065  1.377607e-04
10478   SPP1  4.364415  1.198289e-35
Miss {'CD68'}


In [14]:
from scipy import stats

lfc = sub["log2FC"]

n_up, n_down, n = (lfc > 0).sum(), (lfc < 0).sum(), len(lfc)
print("mean log2FC:", round(lfc.mean(), 3))
print("median     :", round(lfc.median(), 3))
print(f"up/down    : {n_up} / {n_down}")

p = stats.binomtest(n_up, n, 0.5).pvalue      # sign test
print("sign test p:", round(p, 3))

mean log2FC: 0.01
median     : 0.041
up/down    : 6 / 5
sign test p: 1.0


In [15]:

out = df[df["gene"].isin(panel + refs)].copy()         
out["passes"] = (out["p_adj"] < 0.001) & (out["log2FC"].abs() > 0.585)

cols = ["gene", "probe", "log2FC", "p_adj", "status", "passes"]
out = out[cols].sort_values("log2FC")                  

out.to_csv("../results/antigen_presentation_panel.csv", index=False)   # ghi ra file
print(out)

        gene        probe    log2FC         p_adj status  passes
8763   PTPRC  212587_s_at -0.655065  1.377607e-04   Down    True
5290    IRF1    202531_at -0.536591  1.532113e-05     NS   False
945      B2M  216231_s_at -0.200713  3.638519e-04     NS   False
4770   HLA-C  208812_x_at -0.169329  2.009075e-03     NS   False
4769   HLA-B  209140_x_at -0.156182  5.290955e-02     NS   False
4768   HLA-A  215313_x_at -0.068573  3.808882e-01     NS   False
1753    CD8A    205758_at -0.042289  7.006340e-01     NS   False
8663   PSMB9    204279_at  0.040530  8.272163e-01     NS   False
10824  TAPBP    208829_at  0.096526  3.131294e-01     NS   False
10823   TAP2  204769_s_at  0.170167  3.673083e-02     NS   False
10822   TAP1  202307_s_at  0.176616  2.161338e-01     NS   False
8662   PSMB8  209040_s_at  0.250677  2.805001e-02     NS   False
10619  STAT1  200887_s_at  0.507444  4.112118e-05     NS   False
6624   MMP12    204580_at  2.377325  2.587063e-18     Up    True
6621    MMP1    204475_at